In [1]:
!pip install requests pandas

In [2]:
import requests
import pandas as pd
import json

In [3]:
API_KEY = "f82b0388-2e7e-4d18-8207-6fe535c64cb8"
url = "https://api.openchargemap.io/v3/poi/"
params = {
    "output": "json",
    "countrycode": "IN",
    "maxresults": 5000,
    "compact": False,
    "verbose": False,
    "key": API_KEY
}
response = requests.get(url, params=params)
print("Status code:", response.status_code)

Status code: 200


In [4]:
data = response.json()
print("Number of charging stations pulled:", len(data))

Number of charging stations pulled: 1966


In [5]:
data[0]

{'DataProvider': {'WebsiteURL': 'http://openchargemap.org',
  'DataProviderStatusType': {'IsProviderEnabled': True,
   'ID': 1,
   'Title': 'Manual Data Entry'},
  'IsRestrictedEdit': False,
  'IsOpenDataLicensed': True,
  'IsApprovedImport': True,
  'License': 'Licensed under Creative Commons Attribution 4.0 International (CC BY 4.0)',
  'ID': 1,
  'Title': 'Open Charge Map Contributors'},
 'OperatorInfo': {'ID': 1, 'Title': '(Unknown Operator)'},
 'UsageType': {'IsPayAtLocation': False,
  'IsMembershipRequired': False,
  'IsAccessKeyRequired': False,
  'ID': 6,
  'Title': 'Private - For Staff, Visitors or Customers'},
 'StatusType': {'IsOperational': True,
  'IsUserSelectable': True,
  'ID': 50,
  'Title': 'Operational'},
 'SubmissionStatus': {'IsLive': True,
  'ID': 200,
  'Title': 'Submission Published'},
 'IsRecentlyVerified': True,
 'DateLastVerified': '2026-09-19T10:39:00Z',
 'ID': 509422,
 'UUID': '84A0F605-70B0-482E-96F2-0A64CB4EF978',
 'DataProviderID': 1,
 'OperatorID': 1,
 

In [6]:
# ============================================
# CELL 5: Extract data using json_normalize (simpler approach)
# ============================================

# Step 1: Flatten everything except Connections automatically
df_chargers = pd.json_normalize(data)

print("Shape after normalize:", df_chargers.shape)
print("\nAll column names:")
print(df_chargers.columns.tolist())

Shape after normalize: (1966, 72)

All column names:
['IsRecentlyVerified', 'DateLastVerified', 'ID', 'UUID', 'DataProviderID', 'OperatorID', 'UsageTypeID', 'Connections', 'NumberOfPoints', 'StatusTypeID', 'DateLastStatusUpdate', 'DataQualityLevel', 'DateCreated', 'SubmissionStatusTypeID', 'DataProvider.WebsiteURL', 'DataProvider.DataProviderStatusType.IsProviderEnabled', 'DataProvider.DataProviderStatusType.ID', 'DataProvider.DataProviderStatusType.Title', 'DataProvider.IsRestrictedEdit', 'DataProvider.IsOpenDataLicensed', 'DataProvider.IsApprovedImport', 'DataProvider.License', 'DataProvider.ID', 'DataProvider.Title', 'OperatorInfo.ID', 'OperatorInfo.Title', 'UsageType.IsPayAtLocation', 'UsageType.IsMembershipRequired', 'UsageType.IsAccessKeyRequired', 'UsageType.ID', 'UsageType.Title', 'StatusType.IsOperational', 'StatusType.IsUserSelectable', 'StatusType.ID', 'StatusType.Title', 'SubmissionStatus.IsLive', 'SubmissionStatus.ID', 'SubmissionStatus.Title', 'AddressInfo.ID', 'AddressIn

In [8]:
# ============================================
# CELL 6: Rename important columns to simpler names
# ============================================
df_chargers = df_chargers.rename(columns={
    "AddressInfo.Title": "name",
    "AddressInfo.Latitude": "latitude",
    "AddressInfo.Longitude": "longitude",
    "AddressInfo.Town": "town",
    "AddressInfo.StateOrProvince": "state",
    "AddressInfo.Postcode": "postcode",
    "OperatorInfo.Title": "operator",
    "StatusType.Title": "status",
    "StatusType.IsOperational": "is_operational",
    "DateCreated": "date_created",
    "DateLastVerified": "date_last_verified"
})

In [9]:
# CELL 7: Extract max power and connector info from the Connections column
# ============================================
def get_max_power(connections):
    if not connections or not isinstance(connections, list):
        return None
    powers = [c.get("PowerKW") for c in connections if c.get("PowerKW")]
    return max(powers) if powers else None

def get_total_points(connections):
    if not connections or not isinstance(connections, list):
        return 0
    return sum(c.get("Quantity", 1) or 1 for c in connections)

def has_fast_charging(connections):
    if not connections or not isinstance(connections, list):
        return False
    return any(c.get("Level", {}).get("IsFastChargeCapable", False) for c in connections)

df_chargers["max_power_kw"] = df_chargers["Connections"].apply(get_max_power)
df_chargers["total_points"] = df_chargers["Connections"].apply(get_total_points)
df_chargers["has_fast_charging"] = df_chargers["Connections"].apply(has_fast_charging)

In [10]:
# CELL 8: Keep only the columns you actually need
# ============================================
data = df_chargers[[
    "ID", "name", "latitude", "longitude", "town", "state", "postcode",
    "operator", "status", "is_operational", "max_power_kw", 
    "total_points", "has_fast_charging", "date_created", "date_last_verified"
]]

print("Final shape:", data.shape)
data.head()

Final shape: (1966, 15)


,ID,name,latitude,longitude,town,state,postcode,operator,status,is_operational,max_power_kw,total_points,has_fast_charging,date_created,date_last_verified
0,509422,Narketpally - Addanki - Medarametla Road,16.890620,79.561206,MIRYALGUDA,Telangana,508207,(Unknown Operator),Operational,True,180.0,6,True,2026-09-16T10:37:00Z,2026-09-19T10:39:00Z
1,508929,Rajahmundry unit -1,17.039631,81.615838,Rajahmundry,NaN,534313,(Business Owner at Location),Operational,True,240.0,7,True,2026-09-11T08:38:00Z,2026-09-20T04:18:00Z
2,508928,Kakinada Unit 1,16.982049,82.246026,Kakinada,NaN,533003,(Business Owner at Location),Operational,True,60.0,2,True,2026-09-11T08:38:00Z,2026-09-20T04:18:00Z
3,508927,May Fair Green Mark Project,17.484253,78.212304,,NaN,,Salzer NexCharge (IN),Operational,True,7.0,3,False,2026-09-11T08:37:00Z,2026-09-20T04:18:00Z
4,508926,May Fair Marketing Office,17.483787,78.215093,,NaN,,Salzer NexCharge (IN),Operational,True,7.0,1,False,2026-09-11T08:37:00Z,2026-09-20T04:18:00Z


In [11]:
# CELL 9: Save to your data folder
# ============================================
data.to_csv("../data/ev_charging_stations.csv", index=False)
print("Saved successfully! Check your /data folder.")

Saved successfully! Check your /data folder.
